In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Panimula sa Qiskit AI-powered transpiler service

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tinatayang QPU usage: Wala (TANDAAN: Ang tutorial na ito ay hindi nagsasagawa ng mga jobs dahil nakatuon ito sa transpilation)*

## Background
Ang **Qiskit AI-powered transpiler service (QTS)** ay nagpapakilala ng mga optimization na batay sa machine learning sa routing at synthesis passes. Ang mga AI mode na ito ay idinisenyo upang harapin ang mga limitasyon ng tradisyonal na transpilation, lalo na para sa malalaking circuit at komplikadong hardware topology.

Simula **Hulyo 2025**, ang **Transpiler Service** ay inilipat na sa bagong IBM Quantum&reg; Platform at hindi na available. Para sa pinakabagong updates tungkol sa status ng Transpiler Service, mangyaring sumangguni sa [transpiler service documentation](/guides/qiskit-transpiler-service). Maaari pa rin ninyong gamitin ang AI transpiler nang lokal, katulad ng standard na Qiskit transpilation. Palitan lang ang `generate_preset_pass_manager()` ng `generate_ai_pass_manager()`. Ang function na ito ay bumubuo ng pass manager na nagsasama ng AI-powered routing at synthesis passes nang direkta sa inyong lokal na transpilation workflow.

### Mga pangunahing feature ng AI passes
- Routing passes: Ang AI-powered routing ay maaaring dynamic na mag-adjust ng mga qubit path batay sa partikular na circuit at backend, binabawasan ang pangangailangan para sa sobrang SWAP gates.
    - `AIRouting`: Layout selection at circuit routing

- Synthesis passes: Ang mga AI technique ay nag-optimize ng decomposition ng multi-qubit gates, binabawasan ang bilang ng two-qubit gates, na karaniwang mas madaling magkaroon ng error.
    - `AICliffordSynthesis`: Clifford gate synthesis
    - `AILinearFunctionSynthesis`: Linear function circuit synthesis
    - `AIPermutationSynthesis`: Permutation circuit synthesis
    - `AIPauliNetworkSynthesis`: Pauli Network circuit synthesis (available lamang sa Qiskit Transpiler Service, hindi sa lokal na environment)

- Paghahambing sa tradisyonal na transpilation: Ang standard na Qiskit transpiler ay isang matibay na tool na kayang hawakan ang malawak na hanay ng quantum circuits nang epektibo. Gayunpaman, kapag ang mga circuit ay lumalaki o ang mga hardware configuration ay nagiging mas kumplikado, ang mga AI passes ay maaaring magbigay ng karagdagang optimization gains. Sa pamamagitan ng paggamit ng learned models para sa routing at synthesis, ang QTS ay higit pang nagpapabuti ng mga circuit layout at binabawasan ang overhead para sa mahirap o malalaking quantum tasks.

Sinusuri ng tutorial na ito ang mga AI mode gamit ang routing at synthesis passes, inihahambing ang mga resulta sa tradisyonal na transpilation upang i-highlight kung saan nag-aalok ng performance gains ang AI.

Para sa higit pang detalye tungkol sa available na AI passes, tingnan ang [AI passes documentation](/guides/ai-transpiler-passes).

### Bakit gumamit ng AI para sa quantum circuit transpilation?
Habang lumalaki ang laki at komplikadong ng mga quantum circuit, ang tradisyonal na mga pamamaraan ng transpilation ay nahihirapang mag-optimize ng mga layout at bawasan ang gate counts nang epektibo. Ang mas malalaking circuit, lalo na ang mga kinabibilangan ng daan-daang qubit, ay nagdudulot ng makabuluhang hamon sa routing at synthesis dahil sa mga device constraint, limitadong connectivity, at qubit error rates.

Dito nag-aalok ng potensyal na solusyon ang AI-powered transpilation. Sa pamamagitan ng paggamit ng machine learning techniques, ang AI-powered transpiler sa Qiskit ay maaaring gumawa ng mas matalinong mga desisyon tungkol sa qubit routing at gate synthesis, na humahantong sa mas magandang optimization ng malalaking quantum circuits.

### Maikling benchmarking results
![Graph showing AI transpiler performance against Qiskit](../docs/images/tutorials/ai-transpiler-introduction/ai-transpiler-benchmarks.avif)

Sa mga benchmarking test, ang AI transpiler ay patuloy na gumawa ng mas mababaw at mas mataas na kalidad na mga circuit kumpara sa standard na Qiskit transpiler. Para sa mga test na ito, ginamit namin ang default pass manager strategy ng Qiskit, na naka-configure sa [`generate_preset_passmanager`]. Bagama't ang default strategy na ito ay madalas na epektibo, maaari itong mahirapan sa mas malalaki o mas kumplikadong circuit. Sa kabaligtaran, ang AI-powered passes ay nakamit ang average na 24% reduction sa two-qubit gate counts at 36% reduction sa circuit depth para sa malalaking circuit (100+ qubits) kapag nag-transpile sa heavy-hex topology ng IBM Quantum hardware. Para sa higit pang impormasyon tungkol sa mga benchmark na ito, sumangguni sa [blog](https://www.ibm.com/quantum/blog/qiskit-performance) na ito.

Sinusuri ng tutorial na ito ang mga pangunahing benepisyo ng AI passes at kung paano ito inihahambing sa tradisyonal na mga pamamaraan.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.random import random_circuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
import matplotlib.pyplot as plt
from statistics import mean, stdev
import time
import logging

seed = 42


def transpile_with_metrics(pass_manager, circuit):
    """Transpile a circuit and return the result along with key metrics."""
    start = time.time()
    qc_out = pass_manager.run(circuit)
    elapsed = time.time() - start

    depth_2q = qc_out.depth(lambda x: x.operation.num_qubits == 2)
    gate_count = qc_out.size()

    return qc_out, {
        "depth_2q": depth_2q,
        "gate_count": gate_count,
        "time_s": round(elapsed, 3),
    }


def remap_to_contiguous(tqc):
    """Remap a transpiled circuit to use contiguous qubit indices.

    Transpiled circuits target specific physical qubits (e.g., qubit 45, 67)
    on a large backend. This remaps them to 0, 1, 2, ... so Aer only
    simulates the active qubits."""
    active = sorted(
        {tqc.find_bit(q).index for inst in tqc.data for q in inst.qubits}
    )
    qubit_map = {old: new for new, old in enumerate(active)}
    new_qc = QuantumCircuit(len(active))
    for inst in tqc.data:
        old_indices = [tqc.find_bit(q).index for q in inst.qubits]
        new_qc.append(inst.operation, [qubit_map[i] for i in old_indices])
    return new_qc


def build_mirror_circuit(tqc, simulate=True):
    """Build a mirror circuit: U followed by U-dagger, with measurements.

    The expected output is always |0...0>, so measuring the survival
    probability reveals how much noise each transpilation strategy adds.

    Args:
        tqc: A transpiled circuit.
        simulate: If True (default), remap to contiguous qubits so Aer
            only simulates the active qubits. If False, keep the full
            physical layout for hardware execution."""
    if simulate:
        tqc = remap_to_contiguous(tqc)
    mirror = tqc.compose(tqc.inverse())
    mirror.measure_all()
    return mirror


def print_summary(results):
    """Print a summary of each metric as mean +/- stdev across all circuits,
    along with the mean percentage improvement of AI over Default."""
    metrics = [
        ("Depth 2Q", "Depth 2Q (Default)", "Depth 2Q (AI)"),
        ("Gate Count", "Gate Count (Default)", "Gate Count (AI)"),
        ("Time (s)", "Time (Default)", "Time (AI)"),
    ]
    header = (
        f"{'Metric':<12}{'Default (mean +/- std)':>24}"
        f"{'AI (mean +/- std)':>22}{'AI % improvement':>22}"
    )
    print(header)
    print("-" * len(header))
    for label, col_def, col_ai in metrics:
        defaults = [r[col_def] for r in results]
        ais = [r[col_ai] for r in results]
        pct = [(d - a) / d * 100 for d, a in zip(defaults, ais)]
        default_str = f"{mean(defaults):.1f} +/- {stdev(defaults):.1f}"
        ai_str = f"{mean(ais):.1f} +/- {stdev(ais):.1f}"
        pct_str = f"{mean(pct):+.1f}% +/- {stdev(pct):.1f}%"
        print(f"{label:<12}{default_str:>24}{ai_str:>22}{pct_str:>22}")


def plot_metrics_and_pct(results, title_prefix):
    """Plot metric comparisons and percentage improvement of AI over Default."""
    qubits = [r["Qubits"] for r in results]
    metrics = [
        ("Depth 2Q (Default)", "Depth 2Q (AI)", "Two-Qubit Depth"),
        ("Gate Count (Default)", "Gate Count (AI)", "Gate Count"),
        ("Time (Default)", "Time (AI)", "Transpilation Time"),
    ]

    # Row 1: raw metric comparison
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: Metric Comparison",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        ax.plot(qubits, [r[col_def] for r in results], "o-", label="Default")
        ax.plot(qubits, [r[col_ai] for r in results], "s-", label="AI")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel(label)
        ax.legend()
    plt.tight_layout()
    plt.show()

    # Row 2: percentage improvement
    fig, axs = plt.subplots(1, 3, figsize=(21, 5))
    fig.suptitle(
        f"{title_prefix}: % Improvement of AI over Default",
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )
    for ax, (col_def, col_ai, label) in zip(axs, metrics):
        pct = [(r[col_def] - r[col_ai]) / r[col_def] * 100 for r in results]
        ax.axhline(
            0, color="#1f77b4", linewidth=2, label="Default (baseline)"
        )
        ax.plot(qubits, pct, "s-", color="#ff7f0e", label="AI")
        ax.fill_between(qubits, 0, pct, alpha=0.15, color="#ff7f0e")
        ax.set_title(label)
        ax.set_xlabel("Number of Qubits")
        ax.set_ylabel("% Improvement")
        ax.legend()
    plt.tight_layout()
    plt.show()


# Suppress verbose AI-powered transpiler logs
logging.getLogger(
    "qiskit_ibm_transpiler.wrappers.ai_local_synthesis"
).setLevel(logging.WARNING)

## Requirements

Bago simulan ang tutorial na ito, siguraduhing mayroon kayong sumusunod na naka-install:

* Qiskit SDK v1.0 o mas bago, na may [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization) support
* Qiskit Runtime (`pip install qiskit-ibm-runtime`) v0.22 o mas bago
* Qiskit IBM&reg; Transpiler with AI local mode(`pip install 'qiskit-ibm-transpiler[ai-local-mode]'`)

## Setup

In [2]:
num_circuits_sim = 20
depth_sim = 4
qubit_range_sim = list(range(6, 26))

circuits_sim = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_sim,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_sim)
]

print(
    f"Created {len(circuits_sim)} circuits with qubit counts: {qubit_range_sim}"
)
circuits_sim[0].draw(output="mpl", fold=-1)

Created 20 circuits with qubit counts: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step1-code-1.avif" alt="Output of the previous code cell" />

# Part I. Qiskit patterns
Tingnan natin ngayon kung paano gamitin ang AI transpiler service sa isang simpleng quantum circuit, gamit ang Qiskit patterns. Ang susi ay ang paglikha ng `PassManager` gamit ang `generate_ai_pass_manager()` sa halip na ang standard na `generate_preset_pass_manager()`.
## Step 1: Map classical inputs to a quantum problem
Sa seksyong ito, susubukan natin ang AI transpiler sa `efficient_su2` circuit, isang malawakang ginagamit na hardware-efficient ansatz. Ang circuit na ito ay partikular na nauugnay para sa variational quantum algorithms (halimbawa, VQE) at quantum machine-learning tasks, ginagawa itong perpektong test case para sa pagsusuri ng transpilation performance.

Ang `efficient_su2` circuit ay binubuo ng alternating layers ng single-qubit rotations at entangling gates tulad ng CNOTs. Ang mga layer na ito ay nagbibigay-daan sa flexible na pagsisiyasat ng quantum state space habang pinapanatiling manageable ang gate depth. Sa pamamagitan ng pag-optimize sa circuit na ito, naglalayong nating bawasan ang gate count, pahusayin ang fidelity, at bawasan ang noise. Ginagawa itong malakas na kandidato para sa pagsubok sa efficiency ng AI transpiler.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    min_num_qubits=100, operational=True, simulator=False
)


pm_default_sim = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

In [4]:
results_sim = []

for i, qc in enumerate(circuits_sim):
    n = qubit_range_sim[i]

    qc_default, m_default = transpile_with_metrics(pm_default_sim, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_sim.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_sim)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q               33.0 +/- 12.9          26.4 +/- 8.0      +15.8% +/- 17.6%
Gate Count           522.0 +/- 266.0       560.5 +/- 279.1        -9.0% +/- 9.0%
Time (s)                 0.0 +/- 0.0           0.2 +/- 0.1    -893.6% +/- 362.9%


The summary table shows the mean and standard deviation of each metric across all 20 circuits, along with the average percentage improvement of the AI-powered transpiler over the default. Positive values indicate the AI-powered transpiler produced better results; negative values indicate the default was better.

For this small-scale example, the AI-powered transpiler achieves roughly 16% lower two-qubit depth on average, but at the cost of roughly 9% higher gate count. This highlights a key trade-off when choosing between the two strategies: the AI-powered transpiler prioritizes depth reduction (fewer sequential layers of two-qubit gates), while the default transpiler (SABRE) prioritizes minimizing total gate count (fewer SWAP insertions). Depending on your application, one metric may matter more than the other.

In [5]:
plot_metrics_and_pct(results_sim, "Small-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step2-plot-1.avif" alt="Output of the previous code cell" />

### Lumikha ng AI at tradisyonal na pass managers
Upang masuri ang pagiging epektibo ng AI transpiler, magsasagawa tayo ng dalawang transpilation runs. Una, i-transpile natin ang circuit gamit ang AI transpiler. Pagkatapos, magsasagawa tayo ng paghahambing sa pamamagitan ng pag-transpile sa parehong circuit nang walang AI transpiler, gamit ang tradisyonal na mga pamamaraan. Parehong mga proseso ng transpilation ay gagamit ng parehong coupling map mula sa napiling backend at ang optimization level na nakatakda sa 3 para sa patas na paghahambing.

Ang parehong mga pamamaraan na ito ay sumasalamin sa standard na approach para lumikha ng `PassManager` instances upang mag-transpile ng mga circuit sa Qiskit.

In [6]:
# Use the 10-qubit circuit (index where qubits == 10)
idx_10q = qubit_range_sim.index(10)

qc_10q = circuits_sim[idx_10q]
qc_default_10q, _ = transpile_with_metrics(pm_default_sim, qc_10q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_10q, _ = transpile_with_metrics(pm_ai, qc_10q)

tqc_methods = {
    "Default": qc_default_10q,
    "AI": qc_ai_10q,
}

print(
    f"Default: depth {qc_default_10q.depth()}, gates {qc_default_10q.size()}"
)
print(f"AI:      depth {qc_ai_10q.depth()}, gates {qc_ai_10q.size()}")

Default: depth 84, gates 280
AI:      depth 91, gates 343


In [7]:
# Build a simple depolarizing noise model
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.001, 1),
    ["sx", "x", "rz"],  # ~0.1% per 1Q gate
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.01, 2),
    ["cx", "ecr"],  # ~1% per 2Q gate
)

aer_sim = AerSimulator(noise_model=noise_model)

shots = 10000
survival_probs = {}

for method, tqc in tqc_methods.items():
    mirror = build_mirror_circuit(tqc, simulate=True)

    sampler = SamplerV2(mode=aer_sim)
    job = sampler.run([mirror], shots=shots)
    counts = job.result()[0].data.meas.get_counts()

    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots
    survival_probs[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots})"
    )

Default   P(|00...0>) = 0.8460  (8460/10000)
AI        P(|00...0>) = 0.8121  (8121/10000)


We ran both mirror circuits through the Aer simulator with a simple depolarizing noise model. The survival probability, defined as the fraction of shots that return the all-zeros bitstring, quantifies how much noise each transpilation strategy introduces.

### Step 4: Post-process and return result in desired classical format

We extract the probability of measuring the all-zeros bitstring from both runs. A higher survival probability indicates better fidelity, meaning the transpilation introduced less noise. The plot below shows the complement, 1 - P(|0...0>), so that a lower bar indicates better fidelity and small differences in error are easier to see.

In [8]:
# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs = {method: 1 - p for method, p in survival_probs.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs.keys(),
    error_probs.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title("Mirror Circuit Error (10-qubit, Aer Simulator)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/sim-step4-code-0.avif" alt="Output of the previous code cell" />

Sa test na ito, inihahambing natin ang performance ng AI transpiler at ng standard na transpilation method sa efficient_su2 circuit. Ang AI transpiler ay nakakamit ng kapansin-pansing mas mababaw na circuit depth habang pinapanatili ang katulad na gate count.

- **Circuit depth:** Ang AI transpiler ay gumagawa ng circuit na may mas mababang two-qubit depth. Inaasahan ito, dahil ang mga AI passes ay sinanay upang mag-optimize ng depth sa pamamagitan ng pag-aaral ng mga qubit interaction pattern at pagsasamantala ng hardware connectivity nang mas epektibo kaysa sa rule-based heuristics.

- **Gate count:** Ang kabuuang gate count ay nananatiling katulad sa pagitan ng dalawang pamamaraan. Ito ay tumutugma sa mga inaasahan dahil ang standard na SABRE-based transpilation ay tahasang binabawasan ang swap count, na nangingibabaw sa gate overhead. Ang AI transpiler sa halip ay nag-priority sa kabuuang depth at maaaring paminsan-minsan ay mag-trade off ng ilang karagdagang gates para sa mas maikling execution path.

- **Transpilation time:** Ang AI transpiler ay tumatagal ng mas matagal upang tumakbo kaysa sa standard na pamamaraan. Ito ay dahil sa idinagdag na computational cost ng pag-invoke ng learned models sa panahon ng routing at synthesis. Sa kabaligtaran, ang SABRE-based transpiler ay ngayon ay mas mabilis matapos muling isulat at i-optimize sa Rust, nagbibigay ng lubhang efficient na heuristic routing sa scale.

Mahalagang tandaan na ang mga resultang ito ay batay lamang sa isang circuit. Upang makakuha ng komprehensibong pag-unawa kung paano inihahambing ang AI transpiler sa tradisyonal na mga pamamaraan, kinakailangan na subukan ang iba't ibang mga circuit. Ang performance ng QTS ay maaaring mag-iba nang malaki depende sa uri ng circuit na ino-optimize. Para sa mas malawak na paghahambing, sumangguni sa mga benchmark sa itaas o bisitahin ang [blog](https://www.ibm.com/quantum/blog/qiskit-performance).
## Step 3: Execute using Qiskit primitives
Dahil ang tutorial na ito ay nakatuon sa transpilation, walang mga eksperimento na isasagawa sa quantum device. Ang layunin ay samantalahin ang mga optimization mula sa Step 2 upang makakuha ng transpiled circuit na may binawasang depth o gate count.
## Step 4: Post-process and return result in desired classical format
Dahil walang execution para sa notebook na ito, walang mga resultang i-post-process.
# Part II. Analyze and benchmark the transpiled circuits
Sa seksyong ito, ipapakita natin kung paano suriin ang transpiled circuit at i-benchmark ito laban sa orihinal na bersyon nang mas detalyado. Magtutuon tayo sa mga metrics tulad ng circuit depth, gate count, at transpilation time upang masuri ang pagiging epektibo ng optimization. Bukod pa rito, tatalakayin natin kung paano maaaring mag-iba ang mga resulta sa iba't ibang uri ng circuit, nag-aalok ng mga insight sa mas malawak na performance ng transpiler sa iba't ibang sitwasyon.

In [9]:
# -------------------------Step 1-------------------------
num_circuits_hw = 25
depth_hw = 8
qubit_range_hw = list(range(26, 51))

circuits_hw = [
    # We have only two qubit gates, as those test how well the transpiler can optimize the circuit.
    random_circuit(
        num_qubits=n,
        depth=depth_hw,
        max_operands=2,
        num_operand_distribution={2: 1},
        seed=seed + i,
    )
    for i, n in enumerate(qubit_range_hw)
]

print(
    f"Created {len(circuits_hw)} circuits with qubit counts: {qubit_range_hw}"
)

Created 25 circuits with qubit counts: [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


In [10]:
# -------------------------Step 2-------------------------
pm_default = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    seed_transpiler=seed,
)

results_hw = []

for i, qc in enumerate(circuits_hw):
    n = qubit_range_hw[i]

    qc_default, m_default = transpile_with_metrics(pm_default, qc)

    # Create a fresh AI pass manager each iteration to avoid stale layout state
    pm_ai = generate_ai_pass_manager(
        optimization_level=1,
        ai_optimization_level=3,
        backend=backend,
    )
    qc_ai, m_ai = transpile_with_metrics(pm_ai, qc)

    results_hw.append(
        {
            "Qubits": n,
            "Depth 2Q (Default)": m_default["depth_2q"],
            "Depth 2Q (AI)": m_ai["depth_2q"],
            "Gate Count (Default)": m_default["gate_count"],
            "Gate Count (AI)": m_ai["gate_count"],
            "Time (Default)": m_default["time_s"],
            "Time (AI)": m_ai["time_s"],
        }
    )

print_summary(results_hw)

Metric        Default (mean +/- std)     AI (mean +/- std)      AI % improvement
--------------------------------------------------------------------------------
Depth 2Q              217.4 +/- 50.4        191.0 +/- 35.6      +10.9% +/- 10.7%
Gate Count         4513.3 +/- 1394.3     5227.1 +/- 1536.4       -16.4% +/- 5.8%
Time (s)                 0.1 +/- 0.0           3.5 +/- 1.5   -3588.2% +/- 643.6%


In [11]:
plot_metrics_and_pct(results_hw, "Large-Scale Random Circuits")

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step2-plot-1.avif" alt="Output of the previous code cell" />

In [12]:
# -------------------------Step 3-------------------------
# Build mirror circuits from the 26-qubit case
idx_26q = qubit_range_hw.index(26)

qc_26q = circuits_hw[idx_26q]
qc_default_26q, _ = transpile_with_metrics(pm_default, qc_26q)

pm_ai = generate_ai_pass_manager(
    optimization_level=1,
    ai_optimization_level=3,
    backend=backend,
)
qc_ai_26q, _ = transpile_with_metrics(pm_ai, qc_26q)

mirror_default_hw = build_mirror_circuit(qc_default_26q, simulate=False)
mirror_ai_hw = build_mirror_circuit(qc_ai_26q, simulate=False)

# Re-transpile to basis gates (the inverse can introduce gates like sxdg)
pm_basis = generate_preset_pass_manager(
    optimization_level=0,
    backend=backend,
)
mirror_default_hw = pm_basis.run(mirror_default_hw)
mirror_ai_hw = pm_basis.run(mirror_ai_hw)

print(
    f"Mirror circuit (Default): depth {mirror_default_hw.depth()}, gates {mirror_default_hw.size()}"
)
print(
    f"Mirror circuit (AI):      depth {mirror_ai_hw.depth()}, gates {mirror_ai_hw.size()}"
)

# Submit to real hardware
sampler_hw = SamplerV2(mode=backend)
sampler_hw.options.environment.job_tags = ["TUT_AITI"]

shots_hw = 500000
job_hw = sampler_hw.run([mirror_default_hw, mirror_ai_hw], shots=shots_hw)
print(f"Job submitted: {job_hw.job_id()}")

Mirror circuit (Default): depth 1577, gates 9672
Mirror circuit (AI):      depth 1235, gates 11092
Job submitted: d8gt7vm6983c73dqbg0g


In [13]:
# -------------------------Step 4-------------------------
result_hw = job_hw.result()

survival_probs_hw = {}
for i, method in enumerate(["Default", "AI"]):
    counts = result_hw[i].data.meas.get_counts()
    mirror = [mirror_default_hw, mirror_ai_hw][i]
    all_zeros = "0" * mirror.num_qubits
    survival = counts.get(all_zeros, 0) / shots_hw
    survival_probs_hw[method] = survival
    print(
        f"{method:8s}  P(|00...0>) = {survival:.4f}  ({counts.get(all_zeros, 0)}/{shots_hw})"
    )

# Plot 1 - P(|0...0>), the probability of an erroneous (non-zero) outcome.
# A lower bar means the transpilation introduced less noise.
error_probs_hw = {method: 1 - p for method, p in survival_probs_hw.items()}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    error_probs_hw.keys(),
    error_probs_hw.values(),
    color=["steelblue", "coral"],
)
ax.set_ylabel("1 - P(|0...0>)")
ax.set_title(f"Mirror Circuit Error (26-qubit, {backend.name})")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

Default   P(|00...0>) = 0.0005  (239/500000)
AI        P(|00...0>) = 0.0050  (2516/500000)


<Image src="../docs/images/tutorials/ai-transpiler-introduction/extracted-outputs/hw-step4-1.avif" alt="Output of the previous code cell" />

### Analysis of results

The large-scale results reinforce the trends observed in the small-scale example, now at a more demanding scale.

**Two-qubit depth:** The AI-powered transpiler continues to deliver noticeably lower two-qubit depth across the full range of circuit sizes. Depth optimization is one of the primary objectives the AI routing model is trained on, and the advantage is more pronounced at larger qubit counts where the routing problem becomes harder for heuristic methods.

**Gate count:** The default transpiler (SABRE) consistently produces circuits with fewer gates across all circuit sizes in this range. SABRE's heuristic is specifically designed to minimize gate count, and at this scale the advantage is clear and uniform.

**Transpilation time:** The gap in transpilation time widens at larger scales. SABRE remains nearly constant, while the AI-powered transpiler's runtime grows more steeply. Despite this, the AI-powered transpiler runtime remains practical for most workflows.

**Mirror circuit fidelity:** Both methods produce survival probabilities well under 1% at this scale, leaving little usable signal. With total gate counts around 10,000 and two-qubit depths exceeding 1,000, the depolarizing noise accumulated across the mirror circuit overwhelms most of the signal. This highlights a key limitation of the mirror circuit approach: while it is simple and requires no classical simulation, it does not scale well to large or deep circuits, where both methods are pushed close to the noise floor and the small surviving signal is dominated by accumulated error.

While these results underscore the AI-powered transpiler's effectiveness, it is important to note its limitations. The AI synthesis method is currently only available for certain coupling maps, which may restrict its broader applicability. This constraint should be considered when evaluating its usage in different scenarios.

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Transpilation optimizations with SABRE](/docs/tutorials/transpilation-optimizations-with-sabre)
- [Compilation methods for Hamiltonian simulation circuits](/docs/tutorials/compilation-methods-for-hamiltonian-simulation-circuits)

</Admonition>